# 19. INNER JOIN

*Module: 5 — Multi-table · Companion summary: `lectures/19-inner-join.md` · SQL engine: SQLite (in-memory)*

## What you will study

- define INNER JOIN and what it keeps/drops
- see the fan-out effect of one-to-many matches
- spot the rows an INNER JOIN silently loses

---
## Setup — build the sample database this notebook runs on
*Run the cell below first: it creates an in-memory SQLite database and the `q()` helper that prints every query's result set.*

In [ ]:
import sqlite3
import textwrap

conn = sqlite3.connect(':memory:')  # throw-away in-memory database
conn.execute("PRAGMA foreign_keys = ON")

def q(sql):
    """Run a query, print formatted SQL, and display the result set as a table."""
    clean_sql = textwrap.dedent(sql).strip()

    # 1. Print formatted SQL query
    print("\n" + "=" * 60)
    print("QUERY:")
    print(textwrap.indent(clean_sql, "  "))
    print("-" * 60)

    cur = conn.execute(sql)
    rows = cur.fetchall()

    if cur.description:  # SELECT / statement returning rows
        headers = [d[0] for d in cur.description]
        str_rows = [[("" if v is None else str(v)) for v in r] for r in rows]

        # Calculate dynamic column widths for clean alignment
        widths = [
            max(len(h), *(len(r[i]) for r in str_rows), 1)
            for i, h in enumerate(headers)
        ] if str_rows else [len(h) for h in headers]

        header_line = " | ".join(h.ljust(widths[i]) for i, h in enumerate(headers))
        sep_line = "-+-".join("-" * widths[i] for i in range(len(headers)))

        print(header_line)
        print(sep_line)
        for r in str_rows:
            print(" | ".join(val.ljust(widths[i]) for i, val in enumerate(r)))
        if not str_rows:
            print("(0 rows returned)")
    else:  # DDL / DML (CREATE, INSERT, UPDATE, DELETE)
        affected = cur.rowcount
        print(f"Done (rows affected: {affected})" if affected != -1 else "Done.")

    print("=" * 60)

print('Connection ready:', conn)

## 🧪 Sample data — BORROWER + LOAN (same library as nb18)

In [ ]:
q("CREATE TABLE borrower (borrower_id TEXT PRIMARY KEY, lastname TEXT, country TEXT)")
q("INSERT INTO borrower VALUES ('BR1','Smith','CA'), ('BR2','Ali','IN'), ('BR3','Lee','CN'), ('BR4','Peters','US'), ('BR5','Li','CN'), ('BR6','Wong','CA')")
q("CREATE TABLE loan (borrower_id TEXT REFERENCES borrower(borrower_id), copy_id TEXT, loan_date TEXT)")
q("""
INSERT INTO loan VALUES
    ('BR1','C1','2024-01-10'), ('BR1','C2','2024-02-01'),
    ('BR2','C3','2024-03-05'), ('BR3','C4','2024-04-01')
""")
q("SELECT * FROM borrower")
q("SELECT * FROM loan")

## 🧪 INNER JOIN — only matched pairs survive

Both spellings are identical in SQLite (and most engines):

In [ ]:
q("""
SELECT b.lastname, l.copy_id, l.loan_date
FROM borrower b
INNER JOIN loan l ON b.borrower_id = l.borrower_id
""")

In [ ]:
q("""
SELECT b.lastname, l.copy_id, l.loan_date
FROM borrower b JOIN loan l ON b.borrower_id = l.borrower_id   -- INNER is optional
""")

## 🧪 Two behaviors to notice

1. **Fan-out** — Smith has 2 loans, so Smith appears twice.
2. **Lost rows** — Peters, Li and Wong have no loans: they're *silently dropped*.

In [ ]:
q("SELECT lastname FROM borrower")   # 6 people
q("SELECT DISTINCT b.lastname FROM borrower b JOIN loan l ON b.borrower_id = l.borrower_id")   # 3 people


## 💡 Tips & tricks

- INNER JOIN is a *filter*: any row with no match on the other side disappears.
- Count before/after joins when data completeness matters (fan-out lies about totals:
  4 result rows ≠ 4 borrowers ≠ 4 loans).
- To *find* the lost rows instead of losing them: `LEFT JOIN ... WHERE l.borrower_id
  IS NULL` — the anti-join, covered next lecture (nb20).
- 📕 Note: `JOIN … ON key = key USING(key)` / NATURAL JOIN exist but ON is the
  universal, explicit favorite.

## 🎯 Your turn

1. Borrowers and their loan dates for loans taken in 2024-02 or later (post-join WHERE).
2. Total loan count per country — fan-out included, group the joined result.

In [ ]:
# 💡 Solution
q("SELECT b.lastname, l.loan_date FROM borrower b JOIN loan l ON b.borrower_id = l.borrower_id WHERE l.loan_date >= '2024-02-01'")
q("SELECT b.country, COUNT(*) AS loans FROM borrower b JOIN loan l ON b.borrower_id = l.borrower_id GROUP BY b.country")